In [1]:
from google.colab import drive

print("==> 1. Conectando con Google Drive...")
# Esto abrirá un popup para que des permisos a Colab de leer tu Drive
drive.mount('/content/drive')



Mounted at /content/drive


In [2]:
import os
import sys

# Cambia "Proyecto_XNDL" si le pusiste otro nombre a la carpeta en tu Drive
PROJECT_PATH = '/content/drive/Othercomputers/Mi PC/PRUEBAS/'

print("\n==> 2. Configurando el entorno de trabajo...")
# Si la carpeta no existe, te avisamos para evitar errores crípticos
if not os.path.exists(PROJECT_PATH):
    print(f"⚠️ ERROR: No se encuentra la ruta {PROJECT_PATH}.")
    print("Asegúrate de que la carpeta existe en la raíz de tu Google Drive.")
else:
    # Cambiamos el directorio de ejecución de Colab a tu carpeta
    os.chdir(PROJECT_PATH)
    
    # Añadimos la ruta al PATH de Python para que pueda hacer "from models import PreActResNet18"
    sys.path.append(PROJECT_PATH)
    
    print(f"✅ Entorno configurado correctamente.")
    print(f"📁 Directorio actual: {os.getcwd()}")
    print("\n==> Archivos detectados:")
    !ls -l


==> 2. Configurando el entorno de trabajo...
✅ Entorno configurado correctamente.
📁 Directorio actual: /content/drive/Othercomputers/Mi PC/PRUEBAS

==> Archivos detectados:
total 37
-rw------- 1 root root  1482 Jun 19 16:52 calc.py
drwx------ 2 root root  4096 Jun 19 16:15 dades
drwx------ 2 root root  4096 Jun 22 12:47 model
-rw------- 1 root root 24440 Jun 22 19:49 PRUEBAS.ipynb
-rw------- 1 root root  3022 Jun 18 16:08 README.md


In [20]:
!cd model; python model_Laia_Beyloc_Ilyas_Ouacham.py

Device: cuda

Carregant train.npz a la RAM...
Carregant val.npz a la RAM...
Classes (14): ['apple', 'basketball', 'brain', 'circle', 'clock', 'compass', 'cookie', 'donut', 'face', 'moon', 'potato', 'sun', 'watermelon', 'wheel']
Train: 112,000  |  Val: 28,000

Paràmetres entrenables: 6,687,822

Epoch  Train Loss  Train Acc  Val Loss  Val Acc
───────────────────────────────────────────────────────
OPTIMIZER param groups: 0.001
    1      2.0670    37.96%    1.5992  59.70%
OPTIMIZER param groups: 0.001
  --> Primer epoch: 39.3s. EPOCHS = 14 
OPTIMIZER param groups: 0.0040000000000000036
    3      1.6238    56.83%    1.3855  67.55%
OPTIMIZER param groups: 0.021186965510185757
    4      1.2407    73.00%    1.3324  67.00%
OPTIMIZER param groups: 0.06043987106372991
    5      1.1434    76.53%    1.1575  75.51%
OPTIMIZER param groups: 0.0936487860804932
    6      1.1010    78.27%    1.1407  75.08%
OPTIMIZER param groups: 0.09942874509554134
    7      1.0675    79.58%    1.0226  81.17%
OPT

In [16]:
!cd model; cat m_def.py

# Laboratori XNDL final notat - classificació d'imatges 32x32 en 14 categories
#  millor accuracy aconseguit --> 88.84% train/ 87.12% val en 26 epochs
import os, time, copy, math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np

#  hiperparàmetres 
DATA_DIR    = "../dades/dataset_npz/"
BATCH_SIZE  = 512
EPOCHS      = 200   
LR          = 0.15  
TIME_LIMIT  = 20 * 60
CUTOUT_SIZE = 5    
CUTOUT_PROB = 0.5



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

#  funció per obtenir les dades npz, però modificada respecte la original per problemes de coll d'ampolla de cpu
class NpzDataset(torch.utils.data.Dataset):
    def __init__(self, npz_path):
        print(f"Pre-loading {os.path.basename(npz_path)} into RAM...")
        data = np.load(npz_path)
        
        images_np = data["images"].astype(np.float32) / 255.0
        
       